# Imports and dependencies

In [ ]:
!pip install -q sentence-transformers mwparserfromhell

In [ ]:
import time
import re
import requests
import numpy as np
import pandas as pd
import mwparserfromhell
import torch
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from google.colab import files

# Settings

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo: {device}")
if device == "cpu":
    print("AVISO: no se detectó GPU. Ve a Entorno de ejecución → Cambiar tipo de entorno → GPU.")

In [ ]:
HEADERS = {"User-Agent": "WikiBiasAnalysis/1.0 (research project; candela.welsh@outlook.com)"}

MODEL_NAME          = "all-mpnet-base-v2"
THRESHOLD           = 0.80
HIGH_DENSITY_THRESHOLD = 20
PEAK_SENSITIVITY    = 1.0
MAX_SNAPSHOTS       = None   # None = sin límite
BATCH_SIZE          = 64     # mayor que en CPU porque la GPU lo aguanta
OUTPUT_CSV          = "drift_80_articulos.csv"

NOISE_SECTIONS = {
    "references", "notes", "bibliography", "further reading",
    "external links", "see also", "footnotes",
}

ARTICULOS_POR_CLUSTER = {
    "cluster_0": [
        "Abortion",
        "Anarcho-capitalism",
        "Mormonism",
        "Native Americans in the United States",
        "Family planning",
        "Apartheid",
        "HIV/AIDS denialism",
        "Anti-Americanism",
        "National-anarchism",
        "Unidentified flying object",
        "Veganism",
        "Divorce",
        "People for the Ethical Treatment of Animals",
        "LGBTQ rights by country or territory",
        "Vector Marketing",
        "Christian right",
        "Black supremacy",
        "Criticism of Walmart",
        "Scientology",
        "Euthanasia",
        "Eugenics",
        "Healthcare reform in the United States",
        "Same-sex marriage",
        "Surrogacy",
        "Domestic violence",
        "Religion and LGBTQ people",
        "Feminism",
        "Female genital mutilation",
        "Masculism",
        "Homeopathy",
        "Black Lives Matter",
        "Christian Science",
        "Self-harm",
        "Anti-Christian sentiment",
        "Islamophobia",
        "Assisted suicide",
    ],
    "cluster_1": [
        "Mikhail Gorbachev",
        "Plame affair",
        "Bashar al-Assad",
        "Vladimir Putin",
        "Chinese intelligence activity abroad",
        "Gun control",
        "Salvador Allende",
        "Soviet war crimes",
        "Cuba",
        "Vladimir Lenin",
        "Russian interference in the 2016 United States election",
        "Andrew Tate",
        "Karl Marx",
    ],
    "cluster_2": [
        "Al-Qaeda",
        "NATO",
        "Politics of North Korea",
        "2003 invasion of Iraq",
        "Palestine Liberation Organization",
        "Sharia",
        "Saddam Hussein",
        "Saudi Arabia",
        "1953 Iranian coup d'état",
        "Quran",
        "Anfal campaign",
        "CNN",
        "September 11 attacks",
        "Genocide denial",
        "History of Israel",
        "Hamas",
        "Hezbollah",
    ],
    "cluster_3": [
        "Ronald Reagan",
        "Mother Teresa",
        "Chris Brown",
        "Brad Pitt",
        "Marilyn Manson",
        "Elon Musk",
        "Angelina Jolie",
        "Ozzy Osbourne",
        "Margaret Thatcher",
    ],
    "cluster_4": [
        "Tibet",
        "Holodomor",
        "1963 South Vietnamese coup d'état",
        "Cambodian genocide",
        "Armenian genocide",
    ],
}

# Lista plana con (cluster, title) para iterar
ALL_ARTICLES = [
    (cluster, title)
    for cluster, titles in ARTICULOS_POR_CLUSTER.items()
    for title in titles
]
print(f"Total artículos: {len(ALL_ARTICLES)}")

# API MediaWiki

In [ ]:
def get_revision_history(title: str) -> pd.DataFrame:
    url = "https://en.wikipedia.org/w/api.php"
    params = {
        "action": "query", "titles": title, "prop": "revisions",
        "rvprop": "ids|timestamp|user|size|comment",
        "rvlimit": 500, "rvdir": "newer", "format": "json",
    }
    revisions = []
    while True:
        r = requests.get(url, params=params, headers=HEADERS, timeout=30)
        r.raise_for_status()
        data = r.json()
        page = next(iter(data["query"]["pages"].values()))
        if "missing" in page:
            raise ValueError(f"Artículo no encontrado: '{title}'")
        revisions.extend(page.get("revisions", []))
        if "continue" not in data:
            break
        params["rvcontinue"] = data["continue"]["rvcontinue"]
        time.sleep(0.05)
    return pd.DataFrame(revisions)


def get_revision_at_date(title: str, date: str) -> tuple:
    url = "https://en.wikipedia.org/w/api.php"
    params = {
        "action": "query", "titles": title, "prop": "revisions",
        "rvprop": "ids|timestamp|size",
        "rvlimit": 1, "rvdir": "older", "rvstart": date, "format": "json",
    }
    r = requests.get(url, params=params, headers=HEADERS, timeout=30)
    r.raise_for_status()
    data = r.json()
    page = next(iter(data["query"]["pages"].values()))
    if "missing" in page:
        raise ValueError(f"Artículo no encontrado: '{title}'")
    if "revisions" not in page or len(page["revisions"]) == 0:
        raise ValueError(f"No hay revisión para '{title}' anterior a {date}")
    rev = page["revisions"][0]
    return rev["revid"], rev["timestamp"]


def get_revision_text(revid: int) -> str:
    url = "https://en.wikipedia.org/w/api.php"
    params = {
        "action": "query", "revids": revid, "prop": "revisions",
        "rvprop": "content", "rvslots": "main", "format": "json",
    }
    r = requests.get(url, params=params, headers=HEADERS, timeout=60)
    r.raise_for_status()
    data = r.json()
    page = next(iter(data["query"]["pages"].values()))
    return page["revisions"][0]["slots"]["main"]["*"]

# Snapshots detection

In [ ]:
def compute_edit_score(hist: pd.DataFrame, freq: str) -> pd.Series:
    hist = hist.copy()
    hist["timestamp"] = pd.to_datetime(hist["timestamp"])
    hist["size"] = pd.to_numeric(hist["size"], errors="coerce").fillna(0)
    hist = hist.sort_values("timestamp")
    hist["size_delta"] = hist["size"].diff().abs().fillna(0)
    hist = hist.set_index("timestamp")

    edit_count  = hist["revid"].resample(freq).count()
    size_change = hist["size_delta"].resample(freq).sum()

    def norm(s):
        rng = s.max() - s.min()
        return (s - s.min()) / rng if rng > 0 else s * 0

    return 0.5 * norm(edit_count) + 0.5 * norm(size_change)


def detect_snapshot_dates(title: str) -> list:
    hist = get_revision_history(title)
    hist["timestamp"] = pd.to_datetime(hist["timestamp"])
    hist["size"] = pd.to_numeric(hist["size"], errors="coerce").fillna(0)
    hist = hist.sort_values("timestamp").reset_index(drop=True)

    total_months = max(
        (hist["timestamp"].iloc[-1] - hist["timestamp"].iloc[0]).days / 30, 1
    )
    density = len(hist) / total_months
    freq = "QE" if density >= HIGH_DENSITY_THRESHOLD else "ME"

    score = compute_edit_score(hist, freq)
    threshold = score.mean() + PEAK_SENSITIVITY * score.std()
    peak_periods = score[score > threshold].sort_values(ascending=False)
    peak_dates = [p.strftime("%Y-%m-%dT00:00:00Z") for p in peak_periods.index]

    first_date = str(hist["timestamp"].iloc[0].date()) + "T00:00:00Z"
    today = pd.Timestamp.now(tz="UTC").strftime("%Y-%m-%dT00:00:00Z")
    all_dates = sorted(set([first_date] + peak_dates + [today]))

    if MAX_SNAPSHOTS is not None and len(all_dates) > MAX_SNAPSHOTS:
        top_peaks = [
            str(p.to_period(freq).start_time.date()) + "T00:00:00Z"
            for p in peak_periods.head(MAX_SNAPSHOTS - 2).index
        ]
        all_dates = sorted(set([first_date] + top_peaks + [today]))

    return all_dates, round(density, 1)

# Wikitext cleaning

In [ ]:
def clean_wikitext(raw: str, min_para_len: int = 80) -> list:
    wikicode = mwparserfromhell.parse(raw)
    sections = wikicode.get_sections(include_lead=True, flat=True)
    kept = []
    for section in sections:
        headings = section.filter_headings()
        if headings:
            heading_text = headings[0].title.strip_code().strip().lower()
            if heading_text in NOISE_SECTIONS:
                continue
        kept.append(section.strip_code())

    full_text = "\n\n".join(kept)
    raw_paras = re.split(r'\n{2,}', full_text)

    paragraphs = []
    for p in raw_paras:
        p = p.strip()
        if not p:
            continue
        if re.match(r'^[\|!]', p):
            continue
        p = re.sub(r'\n', ' ', p)
        p = re.sub(r'\b[a-z]{2,3}:[^\s]+', '', p).strip()
        non_ascii = sum(1 for c in p if ord(c) > 127)
        if len(p) > 0 and non_ascii / len(p) > 0.3:
            continue
        if len(p) >= min_para_len:
            paragraphs.append(p)
    return paragraphs

# Drift

In [ ]:
model = SentenceTransformer(MODEL_NAME, device=device)
print(f"Modelo cargado en {device}: {MODEL_NAME}")

In [ ]:
def analyze_drift_pair(title, snap_a, snap_b):
    paras_a = snap_a["paragraphs"]
    paras_b = snap_b["paragraphs"]

    if not paras_a or not paras_b:
        return None

    emb_a = model.encode(paras_a, show_progress_bar=False, batch_size=BATCH_SIZE,
                         convert_to_numpy=True, normalize_embeddings=True)
    emb_b = model.encode(paras_b, show_progress_bar=False, batch_size=BATCH_SIZE,
                         convert_to_numpy=True, normalize_embeddings=True)

    # Con embeddings normalizados, similitud coseno = producto escalar
    sim_matrix = emb_a @ emb_b.T
    a_to_b = sim_matrix.max(axis=1)
    b_to_a = sim_matrix.max(axis=0)

    dropped_mask = a_to_b < THRESHOLD
    added_mask   = b_to_a < THRESHOLD

    return {
        "title":            title,
        "date_a":           snap_a["timestamp"][:10],
        "date_b":           snap_b["timestamp"][:10],
        "n_paras_a":        len(paras_a),
        "n_paras_b":        len(paras_b),
        "sim_a_to_b":       round(float(a_to_b.mean()), 4),
        "sim_b_to_a":       round(float(b_to_a.mean()), 4),
        "asimetria":        round(float(a_to_b.mean() - b_to_a.mean()), 4),
        "eliminados":       int(dropped_mask.sum()),
        "aniadidos":        int(added_mask.sum()),
        "pct_eliminados":   round(dropped_mask.sum() / len(paras_a), 4),
        "pct_aniadidos":    round(added_mask.sum()   / len(paras_b), 4),
    }

# Pipeline

In [ ]:
all_rows = []
errors   = []

for idx, (cluster, title) in enumerate(ALL_ARTICLES, 1):
    print(f"\n[{idx:02d}/{len(ALL_ARTICLES)}] {title}  ({cluster})")

    # ── 1. Detectar snapshots ─────────────────────────────────────────────────
    try:
        snapshot_dates, density = detect_snapshot_dates(title)
        print(f"  densidad={density} edits/mes  →  {len(snapshot_dates)} snapshots")
    except Exception as e:
        print(f"  ERROR (historial): {e}")
        errors.append({"title": title, "cluster": cluster, "fase": "historial", "error": str(e)})
        continue

    # ── 2. Descargar y limpiar snapshots ─────────────────────────────────────
    snapshots = []
    for date in snapshot_dates:
        try:
            revid, timestamp = get_revision_at_date(title, date)
            raw = get_revision_text(revid)
            paragraphs = clean_wikitext(raw)
            snapshots.append({
                "date": date, "revid": revid,
                "timestamp": timestamp, "paragraphs": paragraphs,
            })
            time.sleep(0.05)
        except Exception as e:
            print(f"  AVISO (snapshot {date[:10]}): {e}")

    if len(snapshots) < 2:
        print("  Menos de 2 snapshots válidos, se omite.")
        errors.append({"title": title, "cluster": cluster, "fase": "snapshots", "error": "<2 snapshots"})
        continue

    # ── 3. Calcular drift para pares consecutivos ────────────────────────────
    for i in range(len(snapshots) - 1):
        try:
            result = analyze_drift_pair(title, snapshots[i], snapshots[i + 1])
            if result is None:
                continue
            result["cluster"] = cluster
            result["density"] = density
            all_rows.append(result)
            print(
                f"  {result['date_a']} → {result['date_b']}: "
                f"A→B={result['sim_a_to_b']:.4f}  B→A={result['sim_b_to_a']:.4f}  "
                f"asim={result['asimetria']:+.4f}  "
                f"elim={result['eliminados']}/{result['n_paras_a']}  "
                f"añad={result['aniadidos']}/{result['n_paras_b']}"
            )
        except Exception as e:
            print(f"  ERROR (drift {i}→{i+1}): {e}")
            errors.append({"title": title, "cluster": cluster, "fase": f"drift_{i}", "error": str(e)})

print(f"\nPipeline completado. Intervalos calculados: {len(all_rows)}")
if errors:
    print(f"Errores registrados: {len(errors)}")

## Saving data

In [ ]:
cols = [
    "cluster", "title", "density",
    "date_a", "date_b",
    "n_paras_a", "n_paras_b",
    "sim_a_to_b", "sim_b_to_a", "asimetria",
    "eliminados", "aniadidos",
    "pct_eliminados", "pct_aniadidos",
]

df = pd.DataFrame(all_rows)[cols]
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"Guardado: {OUTPUT_CSV}  ({len(df)} filas, {df['title'].nunique()} artículos)")

# Descargar automáticamente desde Colab
files.download(OUTPUT_CSV)

if errors:
    errors_df = pd.DataFrame(errors)
    errors_df.to_csv("errores_pipeline.csv", index=False, encoding="utf-8-sig")
    files.download("errores_pipeline.csv")

## Summary

Métricas agregadas por artículo para facilitar la selección posterior:  
- `min_sim_a_to_b`: drift máximo observado (menor similitud = mayor cambio)  
- `max_asimetria`: mayor diferencia A→B vs B→A en cualquier intervalo (patrón de eliminación selectiva)  
- `total_eliminados`: párrafos eliminados acumulados en todos los intervalos

In [ ]:
resumen = (
    df.groupby(["cluster", "title", "density"])
    .agg(
        n_intervalos      = ("date_a", "count"),
        min_sim_a_to_b    = ("sim_a_to_b", "min"),
        min_sim_b_to_a    = ("sim_b_to_a", "min"),
        max_asimetria     = ("asimetria", lambda x: x.abs().max()),
        total_eliminados  = ("eliminados", "sum"),
        total_aniadidos   = ("aniadidos", "sum"),
    )
    .reset_index()
    .sort_values(["cluster", "min_sim_a_to_b"])
)

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)
display(resumen)

resumen.to_csv("resumen_por_articulo.csv", index=False, encoding="utf-8-sig")
files.download("resumen_por_articulo.csv")